# Layer orchestrator
---
### Imports

In [0]:
%run ../common/medallion_functions

### Variables

In [0]:
layer = dbutils.widgets.get("layer")
config_table_path = dbutils.widgets.get("config_table_path")
logs_table_path = dbutils.widgets.get("logs_table_path")

In [0]:
print(f"*INFO*: Executing orchestrator for layer: {layer}")

### Load configurations

In [0]:
config_table_df = spark.read.table(config_table_path)\
    .filter(f"layer == '{layer}' AND active_for_load = true")

In [0]:
print(f"*INFO*: Configuration found for layer {layer}:")
display(config_table_df)

### Execution

In [0]:
for priority_step in sorted([row.priority for row in config_table_df.select("priority").distinct().collect()]):
    print(f"""-------------------------------\n*INFO*: Executing entities with priority *{priority_step}*.""")

    for row in config_table_df.filter(f"priority == {priority_step}").collect():
        try:
            print(f"*INFO*: Executing entity *{row.entity_name.upper()}* for layer *{row.layer.upper()}*.")

            nb_parameters = {
                "layer": row.layer,
                "load_pattern": row.load_pattern,
                "entity_name": row.entity_name,
            }

            execution_result = run_notebook(
                row.notebook_path,
                nb_parameters
            )

            print(f"*INFO*: {execution_result}")

            create_log_entry(
                logs_table_path = logs_table_path,
                log_code = 0,
                status = "SUCCESS",
                layer = row.layer,
                entity_name = row.entity_name,
                load_pattern = row.load_pattern,
                message = execution_result
            )
            
        except Exception as exc:
            print(f"*ERROR*: {exc}.")

            create_log_entry(
                logs_table_path = logs_table_path,
                log_code = 1,
                status = "FAILED",
                layer = row.layer,
                entity_name = row.entity_name,
                load_pattern = row.load_pattern,
                message = exc
            )
